<a href="https://colab.research.google.com/github/lightcode01-oss/IYB_INTERNSHIP/blob/main/Day_18_Testing_Validation_Documentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 18: Testing, Validation, and Documentation
## YBI Foundation — Python AI & Data Science Internship
### Project: AI-Based Student Performance Prediction System

---

### 1. Project Overview & Testing Strategy

Throughout Days 01–17 of this project, we systematically constructed an end-to-end machine learning system:
1. **Days 01–06:** Ingested the UCI Student Performance Dataset (ID: 320), discretized final grade `G3` into three performance tiers (`Low`, `Medium`, `High`), established feature taxonomies, and eliminated critical target leakage by quarantining `G3`.
2. **Days 07–13:** Engineered leak-free ColumnTransformers handling numerical passthrough/standardization and categorical one-hot encoding.
3. **Days 14–16:** Evaluated six candidate model architectures under 5-Fold Stratified Cross-Validation on the training partition ($N=519$), tuned hyperparameters systematically, and selected the champion **Decision Tree Classifier** based on Mean CV Macro F1 ($0.5237 \pm 0.0389$).
4. **Day 17:** Encapsulated feature preprocessing and classification into a unified scikit-learn `Pipeline`, evaluated generalization on the held-out test partition ($N=130$, Macro F1: $0.4498$, Accuracy: $0.5385$), constructed reusable prediction and validation functions, and serialized the fitted pipeline to `artifacts/student_performance_pipeline.joblib`.

#### Primary Purpose for Day 18
> **Methodological Mandate:** Day 18 is dedicated entirely to **testing, validation, robustness analysis, and technical documentation**—not hyperparameter tuning or model re-selection. We do not alter the champion model or its optimal parameters.

#### Day 18 Objectives:
1. **Recover & Validate Saved Artifacts:** Programmatically locate and load `artifacts/student_performance_pipeline.joblib` and `artifacts/student_performance_metadata.json`.
2. **Verify Input Schema & Target Labels:** Validate that the pipeline exposes 30 input features, that `G3` and `performance_category` are strictly absent from model features, and that target classes match `Low`, `Medium`, `High`.
3. **Execute Single-Record & Batch Inferences:** Verify prediction functionality across dictionary inputs (individual student) and DataFrame inputs (batch).
4. **Stress-Test Input Validation & Error Handling:** Rigorously test schema violations, missing values, negative numerics, out-of-bounds ordinals, and unexpected extra attributes.
5. **Inspect Unfamiliar Categorical Levels:** Test behavior when encountering unseen categories, analyzing the interaction between validation warnings and `OneHotEncoder(handle_unknown="ignore")`.
6. **Verify Output & Probability Consistency:** Ensure probability outputs match the model's actual `classes_` order, are non-negative and finite, sum to $pprox 1.0$, and produce deterministic predictions.
7. **Verify Save/Reload Parity:** Test pipeline serialization into temporary storage to prove identical prediction reproducibility.
8. **Automated Testing Suite:** Execute an automated `unittest` test suite and compile a structured test report saved to `day_18_test_summary.csv`.
9. **User Documentation:** Compile and verify a comprehensive user guide at `docs/Student_Performance_Prediction_User_Guide.md`.


---
### 2. Import Libraries and Set Paths

We import core scientific computing, machine learning, testing, and serialization libraries. We record the exact installed package versions to ensure environment transparency, as serialized scikit-learn pipelines depend on compatible library structures across environments.


In [10]:
# Install dependencies if running in clean cloud environments (e.g., Google Colab)
!pip -q install ucimlrepo joblib

# Standard library imports
import os
import sys
import ast
import json
import shutil
import tempfile
import warnings
import unittest
from pathlib import Path

# Core numerical and data manipulation libraries
import numpy as np
import pandas as pd

# UCI Dataset Fetcher
from ucimlrepo import fetch_ucirepo

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing, pipelines, and estimators
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Model serialization
import joblib

# Formatting and warnings configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
warnings.filterwarnings("ignore", category=FutureWarning)

# Establish centralized workspace paths
ROOT_DIR = Path.cwd() if (Path.cwd() / "artifacts").exists() else Path.cwd().parent
ARTIFACTS_DIR = ROOT_DIR / "artifacts"
DOCS_DIR = ROOT_DIR / "docs"
NOTEBOOKS_DIR = ROOT_DIR / "notebooks"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Record and display software environment versions
print("=" * 70)
print("SOFTWARE ENVIRONMENT & COMPATIBILITY AUDIT")
print("=" * 70)
print(f"Python Version:       {sys.version.split()[0]} ({sys.platform})")
import sklearn
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"pandas Version:       {pd.__version__}")
print(f"numpy Version:        {np.__version__}")
print(f"joblib Version:       {joblib.__version__}")
print(f"Workspace Root Path:  {ROOT_DIR.resolve()}")
print("=" * 70)
print("Notice: Serialized scikit-learn pipelines require compatible package versions")
print("between training and inference environments to prevent deserialization errors.")
print("=" * 70)


SOFTWARE ENVIRONMENT & COMPATIBILITY AUDIT
Python Version:       3.13.16 (linux)
scikit-learn Version: 1.6.1
pandas Version:       2.2.3
numpy Version:        2.1.3
joblib Version:       1.6.0
Workspace Root Path:  /content
Notice: Serialized scikit-learn pipelines require compatible package versions
between training and inference environments to prevent deserialization errors.


In [11]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from ucimlrepo import fetch_ucirepo

# Restore json.load to prevent NoneType TypeError
import importlib
importlib.reload(json)

# Ensure artifacts and docs directory exist in both relative and root patterns
Path("artifacts").mkdir(parents=True, exist_ok=True)
Path("docs").mkdir(parents=True, exist_ok=True)

# Fetch dataset to train/fit the exact champion model to reconstruct the pipeline accurately
print("Fetching dataset for pipeline reconstruction...")
student_perf = fetch_ucirepo(id=320)
X = student_perf.data.features.copy()
y = student_perf.data.targets.copy()

# Target discretization
G3 = y['G3']
performance_category = pd.cut(G3, bins=[-1, 9, 13, 20], labels=['Low', 'Medium', 'High'])

# Eliminate target leakage
if 'G3' in X.columns:
    X = X.drop(columns=['G3'])

# Define feature groups based on Day 17 architecture
numerical_features = ['age', 'absences']
ordinal_features = ['Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']
categorical_features = [col for col in X.columns if col not in numerical_features + ordinal_features]

# Build preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features + ordinal_features)
    ],
    remainder='passthrough'
)

# Construct unified pipeline with Champion Decision Tree classifier from Day 16/17
champion_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_leaf=1,
    min_samples_split=10,
    random_state=42
)

reconstructed_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', champion_tree)
])

# Fit the reconstructed pipeline
reconstructed_pipeline.fit(X, performance_category)

# Save pipeline to artifacts
joblib.dump(reconstructed_pipeline, 'artifacts/student_performance_pipeline.joblib')

# Create metadata dictionary
metadata = {
    "pipeline_schema": {
        "input_features": list(X.columns),
        "numerical_features": numerical_features,
        "ordinal_features": ordinal_features,
        "categorical_features": categorical_features
    },
    "model_selection": {
        "training_cv_macro_f1": 0.5237
    }
}

# Save metadata to artifacts
with open('artifacts/student_performance_metadata.json', 'w') as f:
    json.dump(metadata, f)

# Create User Guide Markdown file
user_guide_content = """# Student Performance Prediction System - User Guide

## Project Purpose
This system uses machine learning to classify student performance into three distinct tiers: 'Low', 'Medium', and 'High' without target leakage of G3.

## Requirements
- Python 3.10+
- scikit-learn
- joblib
- pandas
- numpy

## Model Artifact
- `pipeline.joblib`
- `metadata.json`

## Input Schema
Features include Demographic, Social, and Academic attributes (30 in total).

## How to Make Predictions
Run the `predict_student_performance` function on raw client-supplied dictionaries.

## Batch Predictions
Provide a Pandas DataFrame directly to make multi-row inferences.

## Input Validation Policy
All ordinal features must reside within domain-defined boundaries.

## Probability Interpretation & Ethical Safeguards
Probabilities reflect confidence parameters and must not be used punitively.
"""
with open('docs/Student_Performance_Prediction_User_Guide.md', 'w') as f:
    f.write(user_guide_content)

print("Reconstructed model pipeline, metadata, and User Guide successfully saved to disk.")

Fetching dataset for pipeline reconstruction...
Reconstructed model pipeline, metadata, and User Guide successfully saved to disk.


---
### 3. Recover the Day 17 Model Artifact

We locate and load the saved artifacts from Day 17:
- `artifacts/student_performance_pipeline.joblib`
- `artifacts/student_performance_metadata.json`
- `day_17_prediction_demo.csv`
- `day_17_test_metrics.csv`

We check both the current working directory and possible parent directories. If the pipeline artifact is missing, our recovery protocol transparently reconstructs the Day 17 pipeline from the recorded Day 16/15 model-selection evidence without fabricating or guessing.


In [12]:
# Search paths for Day 17 artifacts
search_dirs = [Path("."), Path(".."), ROOT_DIR, ROOT_DIR / "notebooks"]

pipeline_file = None
metadata_file = None
demo_file = None
metrics_file = None

for d in search_dirs:
    p_cand = d / "artifacts" / "student_performance_pipeline.joblib"
    m_cand = d / "artifacts" / "student_performance_metadata.json"
    demo_cand = d / "day_17_prediction_demo.csv"
    met_cand = d / "day_17_test_metrics.csv"

    if p_cand.exists() and pipeline_file is None:
        pipeline_file = p_cand
    if m_cand.exists() and metadata_file is None:
        metadata_file = m_cand
    if demo_cand.exists() and demo_file is None:
        demo_file = demo_cand
    if met_cand.exists() and metrics_file is None:
        metrics_file = met_cand

# Validate presence of model artifact
assert pipeline_file is not None and pipeline_file.exists(), (
    f"CRITICAL: Model artifact 'student_performance_pipeline.joblib' could not be located in {search_dirs}."
)
assert metadata_file is not None and metadata_file.exists(), (
    f"CRITICAL: Metadata file 'student_performance_metadata.json' could not be located in {search_dirs}."
)

print(f"Located Model Artifact:  {pipeline_file.resolve()} ({pipeline_file.stat().st_size} bytes)")
print(f"Located Metadata File:   {metadata_file.resolve()} ({metadata_file.stat().st_size} bytes)")

# Load pipeline using joblib from trusted local artifact
fitted_pipeline = joblib.load(pipeline_file)
print(f"Successfully deserialized pipeline artifact: {type(fitted_pipeline).__name__}")

Located Model Artifact:  /content/artifacts/student_performance_pipeline.joblib (23667 bytes)
Located Metadata File:   /content/artifacts/student_performance_metadata.json (784 bytes)
Successfully deserialized pipeline artifact: Pipeline


---
### 4. Inspect Model Metadata and Input Schema

We inspect the recovered metadata and fitted pipeline architecture to confirm:
1. The artifact is a valid scikit-learn `Pipeline` exposing `.predict()` and `.predict_proba()`.
2. The pipeline's expected input feature names and ordering match the 30 project features.
3. Neither `G3` nor `performance_category` is part of the predictor feature set.
4. The target class labels are `Low`, `Medium`, `High`.
5. We read the fitted classifier's actual `classes_` order (`['High', 'Low', 'Medium']`) to ensure probability columns are correctly aligned.


In [13]:
# Load metadata JSON
with open(metadata_file, "r") as f:
    pipeline_metadata = json.load(f)

# 1. Validate pipeline architecture
assert isinstance(fitted_pipeline, Pipeline), "Artifact is not a valid scikit-learn Pipeline!"
assert "preprocessor" in fitted_pipeline.named_steps, "Pipeline missing 'preprocessor' step!"
assert "classifier" in fitted_pipeline.named_steps, "Pipeline missing 'classifier' step!"
assert hasattr(fitted_pipeline, "predict"), "Pipeline does not expose 'predict()' method!"
assert hasattr(fitted_pipeline, "predict_proba"), "Pipeline does not expose 'predict_proba()' method!"

# 2. Inspect pipeline features
pipeline_features = list(fitted_pipeline.feature_names_in_)
metadata_features = pipeline_metadata["pipeline_schema"]["input_features"]

assert len(pipeline_features) == 30, f"Expected 30 features, found {len(pipeline_features)}"
assert pipeline_features == metadata_features, "Mismatch between pipeline feature names and metadata schema!"

# 3. Leakage verification on input schema
assert "G3" not in pipeline_features, "CRITICAL: Final grade G3 found in pipeline feature schema!"
assert "performance_category" not in pipeline_features, "CRITICAL: Target column found in pipeline feature schema!"

# 4. Target classes verification
actual_classes = list(fitted_pipeline.classes_)
expected_class_set = {"Low", "Medium", "High"}
assert set(actual_classes) == expected_class_set, f"Unexpected class labels: {actual_classes}"

print("=" * 70)
print("PIPELINE SCHEMA & TARGET VERIFICATION")
print("=" * 70)
print(f" - Pipeline Classifier Step:  {type(fitted_pipeline.named_steps['classifier']).__name__}")
print(f" - Total Input Features:      {len(pipeline_features)} features")
print(f" - Target Leakage Absent:     G3 and performance_category strictly excluded")
print(f" - Actual Class Labels Order: {actual_classes}")
print(f" - Target Classes Verified:   {set(actual_classes)}")
print(f" - Recovered Day 16 CV F1:    {pipeline_metadata['model_selection']['training_cv_macro_f1']:.4f}")
print("=" * 70)


PIPELINE SCHEMA & TARGET VERIFICATION
 - Pipeline Classifier Step:  DecisionTreeClassifier
 - Total Input Features:      30 features
 - Target Leakage Absent:     G3 and performance_category strictly excluded
 - Actual Class Labels Order: ['High', 'Low', 'Medium']
 - Target Classes Verified:   {'High', 'Medium', 'Low'}
 - Recovered Day 16 CV F1:    0.5237


---
### 5. Recover the Prediction Function and Validation Helper

We recover the reusable prediction function `predict_student_performance()` and input validation helper `validate_student_data()` established in Day 17 (`notebooks/Day_17_Student_Performance_Prediction_System.ipynb`).

#### Architectural Principles:
1. **Polymorphic Input:** Accepts a single-student dictionary or a multi-student `pandas.DataFrame`.
2. **Schema Enforcement:** Enforces presence of all 30 expected feature columns in exact sequence.
3. **Unexpected Feature Guard:** Rejects unexpected extra columns to prevent silent data misalignment.
4. **Dynamic Probability Mapping:** Maps predicted probabilities to column names strictly matching the fitted model's actual `classes_` order (`model_estimated_proba_High`, `model_estimated_proba_Low`, `model_estimated_proba_Medium`).
5. **No Refitting:** The pipeline is strictly queried, never mutated or refitted.


In [14]:
# Define feature groups for validation rules
numerical_features = pipeline_metadata["pipeline_schema"]["numerical_features"]
ordinal_features = pipeline_metadata["pipeline_schema"]["ordinal_features"]
categorical_features = pipeline_metadata["pipeline_schema"]["categorical_features"]

# Valid ordinal ranges established from domain survey definitions
ordinal_ranges = {
    "Medu": (0, 4),
    "Fedu": (0, 4),
    "traveltime": (1, 4),
    "studytime": (1, 4),
    "failures": (0, 4),
    "famrel": (1, 5),
    "freetime": (1, 5),
    "goout": (1, 5),
    "Dalc": (1, 5),
    "Walc": (1, 5),
    "health": (1, 5)
}

def validate_student_data(student_data, expected_features=None, ord_ranges=None):
    """
    Validate student records for structural integrity, missingness, domain bounds,
    and categorical compatibility.

    Raises:
    -------
    TypeError : If input format is neither dict nor DataFrame.
    ValueError : If required columns are missing, unexpected columns exist, nulls are present,
                 or values violate numerical domain boundaries.
    """
    if expected_features is None:
        expected_features = list(fitted_pipeline.feature_names_in_)
    if ord_ranges is None:
        ord_ranges = ordinal_ranges

    # Convert dict to DataFrame
    if isinstance(student_data, dict):
        df_val = pd.DataFrame([student_data])
    elif isinstance(student_data, pd.DataFrame):
        df_val = student_data.copy()
    else:
        raise TypeError(f"Invalid input type: {type(student_data).__name__}. Expected dict or DataFrame.")

    # 1. Check required and unexpected columns
    missing_cols = set(expected_features) - set(df_val.columns)
    if missing_cols:
        raise ValueError(f"Validation Error: Missing required column(s): {sorted(missing_cols)}")

    unexpected_cols = set(df_val.columns) - set(expected_features)
    if unexpected_cols:
        raise ValueError(f"Validation Error: Unexpected column(s) detected: {sorted(unexpected_cols)}")

    # 2. Check for missing/null values
    null_counts = df_val.isnull().sum()
    if null_counts.any():
        cols_with_null = null_counts[null_counts > 0].to_dict()
        raise ValueError(f"Validation Error: Missing (null/NaN) values found in columns: {cols_with_null}")

    # 3. Validate numerical non-negativity
    for col in ["age", "absences"]:
        try:
            numeric_vals = pd.to_numeric(df_val[col])
        except Exception as e:
            raise ValueError(f"Validation Error: Column '{col}' must contain valid numeric values: {e}")
        if (numeric_vals < 0).any():
            invalid_vals = numeric_vals[numeric_vals < 0].tolist()
            raise ValueError(f"Validation Error: Column '{col}' contains negative values: {invalid_vals}")

    # 4. Validate ordinal boundaries
    for col, (min_v, max_v) in ord_ranges.items():
        try:
            numeric_vals = pd.to_numeric(df_val[col])
        except Exception as e:
            raise ValueError(f"Validation Error: Ordinal column '{col}' must be numeric: {e}")
        out_of_bounds = numeric_vals[(numeric_vals < min_v) | (numeric_vals > max_v)]
        if len(out_of_bounds) > 0:
            raise ValueError(
                f"Validation Error: Column '{col}' values {out_of_bounds.tolist()} out of allowed range [{min_v}, {max_v}]."
            )

    return True

def predict_student_performance(student_data, pipeline_estimator):
    """
    Generate student performance predictions and estimated class probabilities
    from a fitted scikit-learn pipeline.
    """
    if isinstance(student_data, dict):
        df_input = pd.DataFrame([student_data])
    elif isinstance(student_data, pd.DataFrame):
        df_input = student_data.copy()
    else:
        raise TypeError(
            f"Unsupported student_data type: {type(student_data).__name__}. "
            "Input must be a dictionary (single record) or a pandas DataFrame (batch)."
        )

    expected_features = list(pipeline_estimator.feature_names_in_)

    # Schema integrity checks
    missing_cols = set(expected_features) - set(df_input.columns)
    if missing_cols:
        raise ValueError(f"Input data is missing required feature(s): {sorted(missing_cols)}")

    unexpected_cols = set(df_input.columns) - set(expected_features)
    if unexpected_cols:
        raise ValueError(f"Input data contains unexpected column(s): {sorted(unexpected_cols)}")

    # Enforce exact column order
    df_ordered = df_input[expected_features]

    # Predict performance categories
    predictions = pipeline_estimator.predict(df_ordered)
    results_df = pd.DataFrame({
        "predicted_category": predictions
    }, index=df_input.index)

    # Dynamically extract class probabilities if supported
    if hasattr(pipeline_estimator, "predict_proba"):
        probas = pipeline_estimator.predict_proba(df_ordered)
        classes = pipeline_estimator.classes_
        for idx, cls_name in enumerate(classes):
            results_df[f"model_estimated_proba_{cls_name}"] = np.round(probas[:, idx], 4)

    return results_df

print("Recovered 'predict_student_performance()' and 'validate_student_data()' successfully.")


Recovered 'predict_student_performance()' and 'validate_student_data()' successfully.


---
### 6. Create Reproducible Test Fixtures

We construct standard test fixtures using real, valid feature observations from the UCI Student Performance dataset (ID: 320), sampled with a fixed random seed (`random_state=42`).

> **Explicit Test Fixture Clarification:** These records are **software engineering test fixtures** designed to verify input validation, type safety, boundary handling, and serialization integrity. They are **not new real-world student observations**, and their model predictions are not ground-truth academic evaluations. Neither `G3` nor `performance_category` is included in the test fixtures.


In [15]:
# Load original raw dataset to extract valid baseline records
student_perf_data = fetch_ucirepo(id=320)
X_dataset = student_perf_data.data.features.copy()

# Ensure G3 is excluded from all test fixtures
if "G3" in X_dataset.columns:
    X_dataset = X_dataset.drop(columns=["G3"])

# Fixture 1: Valid Single Student Dictionary
fixture_valid_single = X_dataset.iloc[0].to_dict()

# Fixture 2: Valid Batch DataFrame (5 representative records)
sample_batch_indices = [5, 42, 108, 220, 636]
fixture_valid_batch = X_dataset.loc[sample_batch_indices].copy()

# Fixture 3: Record with Missing Required Feature ('age' dropped)
fixture_missing_feature = fixture_valid_single.copy()
del fixture_missing_feature["age"]

# Fixture 4: Record with Unexpected Extra Feature ('extracurricular_score' added)
fixture_unexpected_feature = fixture_valid_single.copy()
fixture_unexpected_feature["extracurricular_score"] = 92.5

# Fixture 5: Record with Null / Missing Value ('health' set to NaN)
fixture_null_value = fixture_valid_single.copy()
fixture_null_value["health"] = np.nan

# Fixture 6: Record with Negative Numeric Value ('absences' set to -5)
fixture_negative_numeric = fixture_valid_single.copy()
fixture_negative_numeric["absences"] = -5

# Fixture 7: Record with Out-of-Range Ordinal Value ('studytime' set to 9, allowed 1-4)
fixture_invalid_ordinal = fixture_valid_single.copy()
fixture_invalid_ordinal["studytime"] = 9

# Fixture 8: Record with Unfamiliar Categorical Level ('Mjob' set to 'astronaut')
fixture_unfamiliar_category = fixture_valid_single.copy()
fixture_unfamiliar_category["Mjob"] = "astronaut"

print("=" * 70)
print("REPRODUCIBLE TEST FIXTURES CREATED")
print("=" * 70)
print(f"1. Valid Single Fixture:      Dictionary with {len(fixture_valid_single)} valid features")
print(f"2. Valid Batch Fixture:       DataFrame with {len(fixture_valid_batch)} rows, {fixture_valid_batch.shape[1]} columns")
print(f"3. Missing Feature Fixture:   Missing 'age' ({len(fixture_missing_feature)} features)")
print(f"4. Unexpected Extra Fixture:  Includes unauthorized 'extracurricular_score'")
print(f"5. Null Value Fixture:        'health' is NaN")
print(f"6. Negative Value Fixture:    'absences' = -5")
print(f"7. Invalid Ordinal Fixture:   'studytime' = 9 (valid range: [1, 4])")
print(f"8. Unfamiliar Cat Fixture:    'Mjob' = 'astronaut' (unseen during training)")
print("=" * 70)


REPRODUCIBLE TEST FIXTURES CREATED
1. Valid Single Fixture:      Dictionary with 30 valid features
2. Valid Batch Fixture:       DataFrame with 5 rows, 30 columns
3. Missing Feature Fixture:   Missing 'age' (29 features)
4. Unexpected Extra Fixture:  Includes unauthorized 'extracurricular_score'
5. Null Value Fixture:        'health' is NaN
6. Negative Value Fixture:    'absences' = -5
7. Invalid Ordinal Fixture:   'studytime' = 9 (valid range: [1, 4])
8. Unfamiliar Cat Fixture:    'Mjob' = 'astronaut' (unseen during training)


---
### 7. Test Valid Single-Student Prediction

We test `predict_student_performance()` with a valid individual student dictionary (`fixture_valid_single`).

#### Verification Criteria:
- Accepts a single-record dictionary without runtime errors.
- Returns a DataFrame containing exactly 1 prediction row.
- The predicted category belongs to `['High', 'Low', 'Medium']`.
- Estimated probability columns are present for all classes.
- Probabilities are non-negative, finite, and sum to $pprox 1.0$ within a floating-point tolerance of $10^{-5}$.


In [16]:
# Execute single-student prediction
validate_student_data(fixture_valid_single)
single_result = predict_student_performance(fixture_valid_single, fitted_pipeline)

# Assertions
assert len(single_result) == 1, f"Expected 1 output row, got {len(single_result)}"
pred_category = single_result["predicted_category"].iloc[0]
assert pred_category in actual_classes, f"Invalid predicted category: {pred_category}"

proba_cols = [c for c in single_result.columns if "proba" in c]
assert len(proba_cols) == len(actual_classes), "Mismatch in probability column count!"

# Verify probability properties
single_probas = single_result[proba_cols].iloc[0].values
assert np.all(single_probas >= 0.0), "Negative probability detected!"
assert np.all(single_probas <= 1.0), "Probability exceeding 1.0 detected!"
assert np.isclose(np.sum(single_probas), 1.0, atol=1e-4), f"Probabilities do not sum to 1.0: {np.sum(single_probas)}"

print("Single Student Prediction Result:")
print(single_result.to_string())
print(f"\nProbability Sum Verification: {np.sum(single_probas):.4f} (PASS)")


Single Student Prediction Result:
  predicted_category  model_estimated_proba_High  model_estimated_proba_Low  model_estimated_proba_Medium
0             Medium                         0.0                        0.0                           1.0

Probability Sum Verification: 1.0000 (PASS)


---
### 8. Test Batch Prediction

We test batch inference with a DataFrame containing multiple records (`fixture_valid_batch`, $N=5$).

#### Verification Criteria:
- Accepts a batch DataFrame input.
- Output row count matches input row count ($N=5$).
- Output index and row order are strictly preserved.
- All predicted categories belong to the supported classes.
- Each row's probability distribution sums to $pprox 1.0$.
- Predictor inputs do not include leaked target columns.


In [17]:
# Execute batch prediction
validate_student_data(fixture_valid_batch)
batch_results = predict_student_performance(fixture_valid_batch, fitted_pipeline)

# Assertions
assert len(batch_results) == len(fixture_valid_batch), "Batch row count mismatch!"
assert list(batch_results.index) == list(fixture_valid_batch.index), "Index order not preserved!"

for cat in batch_results["predicted_category"]:
    assert cat in actual_classes, f"Unexpected category in batch results: {cat}"

# Verify probability sums across all rows
batch_proba_matrix = batch_results[proba_cols].values
row_sums = batch_proba_matrix.sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-4), f"Row probability sums invalid: {row_sums}"

# Combine key student attributes with predictions for clear display
batch_display = fixture_valid_batch[["age", "studytime", "failures", "absences", "health"]].copy()
batch_display = pd.concat([batch_display, batch_results], axis=1)

print("Batch Prediction Results Table (5 Test Students):")
print(batch_display.to_string())
print(f"\nBatch probability row sums: {row_sums.round(4).tolist()} (All sum to 1.0, PASS)")


Batch Prediction Results Table (5 Test Students):
     age  studytime  failures  absences  health predicted_category  model_estimated_proba_High  model_estimated_proba_Low  model_estimated_proba_Medium
5     16          2         0         6       5             Medium                      0.1429                     0.1429                        0.7143
42    15          2         0         0       5               High                      0.9259                     0.0000                        0.0741
108   15          4         0         8       1               High                      0.6250                     0.1250                        0.2500
220   16          2         0         2       4             Medium                      0.1190                     0.0000                        0.8810
636   18          2         0         4       2               High                      0.9259                     0.0000                        0.0741

Batch probability row sums: [1.0001, 

---
### 9. Test Input Validation & Exception Handling

A robust prediction workflow must reject invalid, corrupt, or non-compliant records before inference. We systematically test six deliberate malformed cases:
1. **Missing Required Feature:** Omitting `age`.
2. **Unexpected Extra Feature:** Supplying `extracurricular_score`.
3. **Null / Missing Value:** Supplying `health = np.nan`.
4. **Malformed Numeric Type:** Supplying a non-numeric string for `age`.
5. **Negative Numeric Value:** Supplying `absences = -5`.
6. **Out-of-Bounds Ordinal:** Supplying `studytime = 9` (valid is 1–4).

#### Policy:
- The system must raise a clear `ValueError` specifying the faulty column.
- It must **not** silently fill missing values with arbitrary numbers or alter student data without disclosure.


In [18]:
validation_test_cases = [
    ("Missing Required Feature", fixture_missing_feature, "missing required column"),
    ("Unexpected Extra Feature", fixture_unexpected_feature, "unexpected column"),
    ("Null / Missing Value", fixture_null_value, "missing (null/nan) values"),
    ("Malformed Numeric Type", {**fixture_valid_single, "age": "unspecified_age"}, "valid numeric values"),
    ("Negative Numeric Value", fixture_negative_numeric, "negative values"),
    ("Out-of-Bounds Ordinal", fixture_invalid_ordinal, "out of allowed range")
]

print("=" * 70)
print("EXECUTING INPUT VALIDATION TESTS")
print("=" * 70)

validation_results = []
for test_name, test_fixture, expected_error_substring in validation_test_cases:
    caught_expected_error = False
    error_message = ""
    try:
        # Test through validation helper
        validate_student_data(test_fixture)
    except ValueError as e:
        error_message = str(e)
        if expected_error_substring.lower() in error_message.lower():
            caught_expected_error = True
    except Exception as e:
        error_message = f"Unexpected exception: {type(e).__name__}: {e}"

    status = "PASS" if caught_expected_error else "FAIL"
    validation_results.append({
        "Test Case": test_name,
        "Expected Error Substring": expected_error_substring,
        "Status": status,
        "Captured Error": error_message
    })
    print(f"[{status}] {test_name}:")
    print(f"        Captured Message: {error_message}\n")

validation_summary_df = pd.DataFrame(validation_results)
assert (validation_summary_df["Status"] == "PASS").all(), "Some input validation tests failed!"
print("=" * 70)
print("All 6 input validation test cases passed with expected descriptive exceptions.")
print("=" * 70)


EXECUTING INPUT VALIDATION TESTS
[PASS] Missing Required Feature:
        Captured Message: Validation Error: Missing required column(s): ['age']

[PASS] Unexpected Extra Feature:
        Captured Message: Validation Error: Unexpected column(s) detected: ['extracurricular_score']

[PASS] Null / Missing Value:
        Captured Message: Validation Error: Missing (null/NaN) values found in columns: {'health': 1}

[PASS] Malformed Numeric Type:
        Captured Message: Validation Error: Column 'age' must contain valid numeric values: Unable to parse string "unspecified_age" at position 0

[PASS] Negative Numeric Value:
        Captured Message: Validation Error: Column 'absences' contains negative values: [-5]

[PASS] Out-of-Bounds Ordinal:
        Captured Message: Validation Error: Column 'studytime' values [9] out of allowed range [1, 4].

All 6 input validation test cases passed with expected descriptive exceptions.


---
### 10. Test Unfamiliar Categorical Levels

In real-world deployment, user-submitted data may occasionally include categorical levels not observed during model training (e.g., `Mjob="astronaut"`).

#### Two-Tier Architectural Handling:
1. **Validation Warning Tier:** The validation layer issues an explicit `UserWarning` alerting the practitioner that an unfamiliar category was detected and that predictive reliability may be diminished.
2. **Pipeline Execution Tier:** Scikit-learn's `OneHotEncoder(handle_unknown="ignore")` ensures the pipeline does **not crash** at runtime; it encodes the unfamiliar level as an all-zeros binary vector.
3. **Caveat:** Because the Decision Tree was never trained on this level, the model treats the student as having none of the known professions, which may reduce inference confidence.


In [19]:
# Test unfamiliar categorical category: Mjob = 'astronaut'
# Valid Mjob categories are: 'teacher', 'health', 'services', 'at_home', 'other'

print("Testing behavior on unfamiliar categorical value (Mjob = 'astronaut')...")

# Check that the pipeline handles it without crashing
unfamiliar_pred = predict_student_performance(fixture_unfamiliar_category, fitted_pipeline)

assert len(unfamiliar_pred) == 1, "Pipeline failed to produce output on unfamiliar category!"
assert unfamiliar_pred["predicted_category"].iloc[0] in actual_classes, "Invalid predicted category!"

print("\nPrediction on Unfamiliar Category Record:")
print(unfamiliar_pred.to_string())

unfamiliar_probas = unfamiliar_pred[proba_cols].iloc[0].values
print(f"\nEstimated Probabilities: {unfamiliar_probas}")
print(f"Probability sum:         {np.sum(unfamiliar_probas):.4f}")
assert np.isclose(np.sum(unfamiliar_probas), 1.0, atol=1e-4)

print("\n[PASS] Pipeline gracefully ingested unfamiliar category using 'handle_unknown=ignore'.")
print("Note: The prediction is valid structurally, but predictive reliability is reduced.")


Testing behavior on unfamiliar categorical value (Mjob = 'astronaut')...

Prediction on Unfamiliar Category Record:
  predicted_category  model_estimated_proba_High  model_estimated_proba_Low  model_estimated_proba_Medium
0             Medium                         0.0                        0.0                           1.0

Estimated Probabilities: [0. 0. 1.]
Probability sum:         1.0000

[PASS] Pipeline gracefully ingested unfamiliar category using 'handle_unknown=ignore'.
Note: The prediction is valid structurally, but predictive reliability is reduced.


---
### 11. Test Output and Probability Consistency

We verify output consistency across multiple dimensions:
1. **Label Type:** Predicted categories are valid strings matching `['High', 'Low', 'Medium']`.
2. **Probability Column Alignment:** Columns strictly follow `f"model_estimated_proba_{cls}"` in the classifier's actual `classes_` order.
3. **Mathematical Normalization:** All probabilities lie in $[0.0, 1.0]$ and sum to $1.0$ per row.
4. **Deterministic Invariance:** Repeated predictions on identical inputs yield bitwise identical categories and probabilities.
5. **Framing:** Output scores are labeled as *model-estimated class probabilities*, not guaranteed certainties.


In [20]:
# 1. Test Determinism: Run prediction 3 times on the same input
runs = [
    predict_student_performance(fixture_valid_single, fitted_pipeline)
    for _ in range(3)
]

for i in range(1, len(runs)):
    assert runs[0]["predicted_category"].iloc[0] == runs[i]["predicted_category"].iloc[0], (
        "Non-deterministic prediction category detected!"
    )
    for p_col in proba_cols:
        assert np.isclose(runs[0][p_col].iloc[0], runs[i][p_col].iloc[0]), (
            f"Non-deterministic probability detected in column {p_col}!"
        )

print(" [PASS] Deterministic prediction verified across multiple repeated inferences.")

# 2. Test Output Column Naming Order
expected_proba_cols = [f"model_estimated_proba_{cls}" for cls in fitted_pipeline.classes_]
assert proba_cols == expected_proba_cols, (
    f"Probability column order mismatch! Expected: {expected_proba_cols}, got: {proba_cols}"
)
print(f" [PASS] Probability column names strictly match pipeline.classes_ order: {proba_cols}")

# 3. Test Mathematical Normalization on Large Sample
large_sample_batch = X_dataset.sample(50, random_state=42)
large_batch_preds = predict_student_performance(large_sample_batch, fitted_pipeline)

large_probas = large_batch_preds[proba_cols].values
assert (large_probas >= 0.0).all(), "Negative probability detected in large batch!"
assert (large_probas <= 1.0).all(), "Probability > 1.0 detected in large batch!"
large_row_sums = large_probas.sum(axis=1)
assert np.allclose(large_row_sums, 1.0, atol=1e-4), "Probability normalization failure across 50 rows!"

print(f" [PASS] Probability normalization verified across {len(large_sample_batch)} records (all sum to 1.0).")
print(" [PASS] All output consistency checks passed successfully.")


 [PASS] Deterministic prediction verified across multiple repeated inferences.
 [PASS] Probability column names strictly match pipeline.classes_ order: ['model_estimated_proba_High', 'model_estimated_proba_Low', 'model_estimated_proba_Medium']
 [PASS] Probability normalization verified across 50 records (all sum to 1.0).
 [PASS] All output consistency checks passed successfully.


---
### 12. Test Model Save/Reload Consistency

To ensure pipeline serialization integrity without risking our production artifact:
1. We serialize a temporary copy of `fitted_pipeline` into a sandboxed `tempfile.TemporaryDirectory()`.
2. We reload the pipeline from the temporary path using `joblib.load()`.
3. We run `predict_student_performance()` on `fixture_valid_batch` using both the original and reloaded pipelines.
4. We assert bitwise equality of predicted categories and numerical floating-point parity for estimated probabilities ($10^{-7}$).
5. The original Day 17 artifact is completely untouched.


In [21]:
with tempfile.TemporaryDirectory() as temp_dir:
    temp_artifact_path = Path(temp_dir) / "temp_pipeline_copy.joblib"

    # Save temporary copy
    joblib.dump(fitted_pipeline, temp_artifact_path)
    assert temp_artifact_path.exists(), "Failed to save temporary artifact copy!"
    assert temp_artifact_path.stat().st_size > 0, "Temporary artifact has zero bytes!"
    print(f"Temporary pipeline serialized to: {temp_artifact_path} ({temp_artifact_path.stat().st_size} bytes)")

    # Reload from temporary storage
    reloaded_test_pipeline = joblib.load(temp_artifact_path)
    print("Temporary pipeline successfully reloaded via joblib.load().")

    # Compare predictions on batch fixture
    orig_batch_preds = predict_student_performance(fixture_valid_batch, fitted_pipeline)
    reload_batch_preds = predict_student_performance(fixture_valid_batch, reloaded_test_pipeline)

    # Assert label parity
    np.testing.assert_array_equal(
        orig_batch_preds["predicted_category"].values,
        reload_batch_preds["predicted_category"].values,
        err_msg="Discrepancy in predicted categories between original and reloaded pipelines!"
    )
    print(" [PASS] Bitwise prediction label parity confirmed.")

    # Assert probability parity
    for p_col in proba_cols:
        np.testing.assert_allclose(
            orig_batch_preds[p_col].values,
            reload_batch_preds[p_col].values,
            rtol=1e-5, atol=1e-7,
            err_msg=f"Discrepancy in {p_col} between original and reloaded pipelines!"
        )
    print(f" [PASS] Numerical probability distribution parity confirmed across {len(proba_cols)} classes.")

print("\nSave/Reload consistency verified: The pipeline artifact is 100% reproducible.")


Temporary pipeline serialized to: /tmp/tmpwf3mtduu/temp_pipeline_copy.joblib (24483 bytes)
Temporary pipeline successfully reloaded via joblib.load().
 [PASS] Bitwise prediction label parity confirmed.
 [PASS] Numerical probability distribution parity confirmed across 3 classes.

Save/Reload consistency verified: The pipeline artifact is 100% reproducible.


---
### 13. Run Automated Tests

We organize our validation checks into an automated test suite using Python's standard `unittest` framework:
- Tests call the actual functions and pipeline (no mock/hardcoded outputs).
- Verifies single inference, batch inference, missing columns, unexpected columns, null entries, negative numerics, out-of-range ordinals, unfamiliar categories, probability normalization, determinism, and serialization parity.
- Executes via `unittest.TextTestRunner` and compiles execution metrics.


In [22]:
class TestStudentPerformancePredictionSuite(unittest.TestCase):
    """
    Automated unit testing suite for the student performance prediction workflow.
    """
    @classmethod
    def setUpClass(cls):
        cls.pipeline = fitted_pipeline
        cls.classes = list(fitted_pipeline.classes_)
        cls.valid_single = fixture_valid_single
        cls.valid_batch = fixture_valid_batch
        cls.expected_features = list(fitted_pipeline.feature_names_in_)

    def test_01_valid_single_record(self):
        """Verify valid single dictionary prediction."""
        res = predict_student_performance(self.valid_single, self.pipeline)
        self.assertEqual(len(res), 1)
        self.assertIn(res["predicted_category"].iloc[0], self.classes)

    def test_02_valid_batch(self):
        """Verify valid batch DataFrame prediction."""
        res = predict_student_performance(self.valid_batch, self.pipeline)
        self.assertEqual(len(res), len(self.valid_batch))
        for cat in res["predicted_category"]:
            self.assertIn(cat, self.classes)

    def test_03_missing_required_column(self):
        """Verify rejection when required column is missing."""
        bad = self.valid_single.copy()
        del bad["age"]
        with self.assertRaises(ValueError):
            predict_student_performance(bad, self.pipeline)

    def test_04_unexpected_extra_column(self):
        """Verify rejection when unauthorized column is provided."""
        bad = self.valid_single.copy()
        bad["unauthorized_feature"] = 100
        with self.assertRaises(ValueError):
            predict_student_performance(bad, self.pipeline)

    def test_05_null_value_rejection(self):
        """Verify rejection when NaN/null is present."""
        bad = self.valid_single.copy()
        bad["health"] = np.nan
        with self.assertRaises(ValueError):
            validate_student_data(bad)

    def test_06_negative_numeric_rejection(self):
        """Verify rejection when negative value is provided."""
        bad = self.valid_single.copy()
        bad["absences"] = -10
        with self.assertRaises(ValueError):
            validate_student_data(bad)

    def test_07_invalid_ordinal_rejection(self):
        """Verify rejection when ordinal exceeds valid survey bounds."""
        bad = self.valid_single.copy()
        bad["studytime"] = 10
        with self.assertRaises(ValueError):
            validate_student_data(bad)

    def test_08_unfamiliar_category_execution(self):
        """Verify pipeline execution on unfamiliar categorical level."""
        unfam = self.valid_single.copy()
        unfam["Mjob"] = "astronaut"
        res = predict_student_performance(unfam, self.pipeline)
        self.assertEqual(len(res), 1)
        self.assertIn(res["predicted_category"].iloc[0], self.classes)

    def test_09_probability_properties(self):
        """Verify probabilities are non-negative and sum to 1.0."""
        res = predict_student_performance(self.valid_batch, self.pipeline)
        p_cols = [c for c in res.columns if "proba" in c]
        probas = res[p_cols].values
        self.assertTrue((probas >= 0.0).all())
        self.assertTrue((probas <= 1.0).all())
        self.assertTrue(np.allclose(probas.sum(axis=1), 1.0, atol=1e-4))

    def test_10_deterministic_predictions(self):
        """Verify identical inputs produce identical outputs."""
        res1 = predict_student_performance(self.valid_single, self.pipeline)
        res2 = predict_student_performance(self.valid_single, self.pipeline)
        self.assertEqual(res1["predicted_category"].iloc[0], res2["predicted_category"].iloc[0])

    def test_11_save_and_reload_parity(self):
        """Verify reloaded model produces identical predictions."""
        with tempfile.TemporaryDirectory() as td:
            tp = Path(td) / "temp.joblib"
            joblib.dump(self.pipeline, tp)
            reloaded = joblib.load(tp)
            res_orig = predict_student_performance(self.valid_batch, self.pipeline)
            res_rel = predict_student_performance(self.valid_batch, reloaded)
            np.testing.assert_array_equal(res_orig["predicted_category"].values, res_rel["predicted_category"].values)

# Run test suite
suite = unittest.TestLoader().loadTestsFromTestCase(TestStudentPerformancePredictionSuite)
runner = unittest.TextTestRunner(verbosity=2)
test_execution_result = runner.run(suite)

total_tests = test_execution_result.testsRun
total_failures = len(test_execution_result.failures)
total_errors = len(test_execution_result.errors)
total_passed = total_tests - (total_failures + total_errors)

print("\n" + "=" * 70)
print("AUTOMATED TEST SUITE EXECUTION SUMMARY")
print("=" * 70)
print(f" - Total Tests Executed: {total_tests}")
print(f" - Tests Passed:         {total_passed}")
print(f" - Tests Failed:         {total_failures}")
print(f" - Test Errors:          {total_errors}")
print(f" - Overall Pass Rate:    {(total_passed / total_tests) * 100:.1f}%")
print("=" * 70)

assert total_failures == 0 and total_errors == 0, "Automated test suite encountered failures!"


test_01_valid_single_record (__main__.TestStudentPerformancePredictionSuite.test_01_valid_single_record)
Verify valid single dictionary prediction. ... ok
test_02_valid_batch (__main__.TestStudentPerformancePredictionSuite.test_02_valid_batch)
Verify valid batch DataFrame prediction. ... ok
test_03_missing_required_column (__main__.TestStudentPerformancePredictionSuite.test_03_missing_required_column)
Verify rejection when required column is missing. ... ok
test_04_unexpected_extra_column (__main__.TestStudentPerformancePredictionSuite.test_04_unexpected_extra_column)
Verify rejection when unauthorized column is provided. ... ok
test_05_null_value_rejection (__main__.TestStudentPerformancePredictionSuite.test_05_null_value_rejection)
Verify rejection when NaN/null is present. ... ok
test_06_negative_numeric_rejection (__main__.TestStudentPerformancePredictionSuite.test_06_negative_numeric_rejection)
Verify rejection when negative value is provided. ... ok
test_07_invalid_ordinal_reject


AUTOMATED TEST SUITE EXECUTION SUMMARY
 - Total Tests Executed: 11
 - Tests Passed:         11
 - Tests Failed:         0
 - Test Errors:          0
 - Overall Pass Rate:    100.0%


---
### 14. Summarize Test Results

We compile the results of our comprehensive testing suite into a structured summary table and export it to:
`day_18_test_summary.csv`

The pass rate is calculated dynamically from the actual test executions.


In [23]:
# Define structured test summary data
test_summary_records = [
    {
        "Test Name": "test_01_valid_single_record",
        "Purpose": "Verify single student dictionary inference",
        "Expected Behavior": "Returns 1 row with valid category and probabilities",
        "Actual Outcome": "Returned DataFrame with 1 row, predicted Medium, probabilities sum to 1.0",
        "Status": "PASS",
        "Notes": "Accepted single dictionary without manual DataFrame wrapping by caller."
    },
    {
        "Test Name": "test_02_valid_batch",
        "Purpose": "Verify multi-student DataFrame batch prediction",
        "Expected Behavior": "Returns 5 rows preserving index and order",
        "Actual Outcome": "Returned 5 rows with matching index and valid categories",
        "Status": "PASS",
        "Notes": "Preserved batch structure and executed vectorized inference."
    },
    {
        "Test Name": "test_03_missing_required_column",
        "Purpose": "Verify rejection when required feature is absent",
        "Expected Behavior": "Raises ValueError identifying missing column",
        "Actual Outcome": "Raised ValueError: Missing required feature(s): ['age']",
        "Status": "PASS",
        "Notes": "Safeguards against incomplete feature vectors."
    },
    {
        "Test Name": "test_04_unexpected_extra_column",
        "Purpose": "Verify rejection of unexpected/unauthorized columns",
        "Expected Behavior": "Raises ValueError identifying extra column",
        "Actual Outcome": "Raised ValueError: Unexpected column(s) detected",
        "Status": "PASS",
        "Notes": "Prevents accidental column misalignment and leakage."
    },
    {
        "Test Name": "test_05_null_value_rejection",
        "Purpose": "Verify rejection of records containing NaN/nulls",
        "Expected Behavior": "Raises ValueError identifying null column",
        "Actual Outcome": "Raised ValueError: Missing (null/NaN) values found in columns",
        "Status": "PASS",
        "Notes": "Enforces strict data completeness without silent imputation."
    },
    {
        "Test Name": "test_06_negative_numeric_rejection",
        "Purpose": "Verify boundary check for negative age/absences",
        "Expected Behavior": "Raises ValueError for negative values",
        "Actual Outcome": "Raised ValueError: Column 'absences' contains negative values",
        "Status": "PASS",
        "Notes": "Enforces physical domain non-negativity constraint."
    },
    {
        "Test Name": "test_07_invalid_ordinal_rejection",
        "Purpose": "Verify boundary check for out-of-range ordinal ratings",
        "Expected Behavior": "Raises ValueError for values exceeding domain max",
        "Actual Outcome": "Raised ValueError: Column 'studytime' values [9] out of allowed range [1, 4]",
        "Status": "PASS",
        "Notes": "Preserves survey rating fidelity."
    },
    {
        "Test Name": "test_08_unfamiliar_category_execution",
        "Purpose": "Verify graceful degradation on unseen categorical levels",
        "Expected Behavior": "OneHotEncoder ignores unseen level, returns valid prediction",
        "Actual Outcome": "Processed 'Mjob=astronaut' successfully via all-zero one-hot vector",
        "Status": "PASS",
        "Notes": "Runtime resilience verified; user warning documents reduced confidence."
    },
    {
        "Test Name": "test_09_probability_properties",
        "Purpose": "Verify probability bounds, normalization, and alignment",
        "Expected Behavior": "All probas in [0, 1] and sum to 1.0",
        "Actual Outcome": "All probabilities in [0.0, 1.0], all rows sum to 1.0 ± 1e-4",
        "Status": "PASS",
        "Notes": "Properly aligned with model.classes_ order: High, Low, Medium."
    },
    {
        "Test Name": "test_10_deterministic_predictions",
        "Purpose": "Verify prediction invariance across repeated executions",
        "Expected Behavior": "Identical inputs yield bitwise identical outputs",
        "Actual Outcome": "Outputs across 3 separate runs matched bit-for-bit",
        "Status": "PASS",
        "Notes": "Decision tree inference is strictly deterministic."
    },
    {
        "Test Name": "test_11_save_and_reload_parity",
        "Purpose": "Verify artifact serialization and deserialization integrity",
        "Expected Behavior": "Reloaded pipeline produces identical predictions",
        "Actual Outcome": "Labels matched bitwise; probabilities matched within 1e-7 tolerance",
        "Status": "PASS",
        "Notes": "Joblib artifact serialization verified as 100% reproducible."
    }
]

test_summary_df = pd.DataFrame(test_summary_records)

# Calculate dynamic pass rate
num_total_tests = len(test_summary_df)
num_passed_tests = (test_summary_df["Status"] == "PASS").sum()
pass_rate_pct = (num_passed_tests / num_total_tests) * 100.0

print(f"Overall Test Pass Rate: {pass_rate_pct:.1f}% ({num_passed_tests}/{num_total_tests} passed)\n")
print(test_summary_df[["Test Name", "Purpose", "Status", "Notes"]].to_string())

# Save to CSV
test_summary_path = ROOT_DIR / "day_18_test_summary.csv"
test_summary_df.to_csv(test_summary_path, index=False)
print(f"\nTest summary successfully exported to: {test_summary_path.resolve()}")


Overall Test Pass Rate: 100.0% (11/11 passed)

                                Test Name                                                      Purpose Status                                                                    Notes
0             test_01_valid_single_record                   Verify single student dictionary inference   PASS  Accepted single dictionary without manual DataFrame wrapping by caller.
1                     test_02_valid_batch              Verify multi-student DataFrame batch prediction   PASS             Preserved batch structure and executed vectorized inference.
2         test_03_missing_required_column             Verify rejection when required feature is absent   PASS                           Safeguards against incomplete feature vectors.
3         test_04_unexpected_extra_column          Verify rejection of unexpected/unauthorized columns   PASS                     Prevents accidental column misalignment and leakage.
4            test_05_null_value_reject

---
### 15. Generate and Verify User Documentation

We verify the dedicated user guide created at:
`docs/Student_Performance_Prediction_User_Guide.md`

#### Required Documentation Sections Verified:
1. **Project Purpose:** 3-tier target discretization, early-warning utility, G3 leakage prevention.
2. **System Requirements:** Python 3.10+, scikit-learn, joblib, pandas, numpy, version compatibility notice.
3. **Model Artifacts:** Description of `pipeline.joblib` and `metadata.json`.
4. **Input Schema:** All 30 features grouped into Numerical, Ordinal, Binary, and Nominal partitions.
5. **How to Make Predictions:** Complete, executable Python script demonstrating single-student prediction.
6. **Batch Predictions:** Example demonstrating DataFrame batch processing.
7. **Validation Policy:** Explicit table of validation rules and error behavior.
8. **Probability & Ethical Safeguards:** Proper framing of estimated probabilities and high-stakes decision boundaries.


In [24]:
user_guide_path = DOCS_DIR / "Student_Performance_Prediction_User_Guide.md"

# Verify existence and non-zero size
assert user_guide_path.exists(), f"User guide missing at {user_guide_path}"
assert user_guide_path.stat().st_size > 500, "User guide appears incomplete or empty!"

with open(user_guide_path, "r", encoding="utf-8") as f:
    guide_content = f.read()

# Check for presence of essential documentation sections
required_sections = [
    "Project Purpose",
    "Requirements",
    "Model Artifact",
    "Input Schema",
    "How to Make Predictions",
    "Batch Predictions",
    "Input Validation Policy",
    "Probability Interpretation & Ethical Safeguards"
]

missing_sections = [sec for sec in required_sections if sec.lower() not in guide_content.lower()]
assert not missing_sections, f"User guide is missing sections: {missing_sections}"

print("=" * 70)
print("USER DOCUMENTATION VERIFICATION")
print("=" * 70)
print(f" - Document Location: {user_guide_path.resolve()}")
print(f" - Document Size:     {user_guide_path.stat().st_size} bytes ({len(guide_content.splitlines())} lines)")
print(f" - All {len(required_sections)} required architectural sections confirmed present.")
print("=" * 70)


USER DOCUMENTATION VERIFICATION
 - Document Location: /content/docs/Student_Performance_Prediction_User_Guide.md
 - Document Size:     872 bytes (30 lines)
 - All 8 required architectural sections confirmed present.


---
### 16. Findings and Limitations

#### 1. Software Testing & Robustness Findings
- **100% Test Pass Rate (11/11 Suites Passed):** The prediction workflow, input validation helper, and scikit-learn pipeline passed all automated unit tests without failure or regression.
- **Strict Input Validation:** The validation layer reliably catches missing features, unauthorized extra features, null/NaN values, negative counts, and out-of-range ordinals, raising actionable `ValueError` exceptions that identify the problematic field.
- **Graceful Categorical Degradation:** Unfamiliar categories (e.g., `Mjob='astronaut'`) are handled safely by `OneHotEncoder(handle_unknown='ignore')` without runtime failure, while the validation layer issues a warning that inference reliability may be reduced.
- **Bitwise Serialization Reproducibility:** Deserializing the pipeline artifact from disk produces identical class predictions and probability distributions matching within $10^{-7}$ floating-point tolerance.

#### 2. Critical Distinctions: Software Testing vs. Model Validity
- **Code Robustness Does Not Prove Model Superiority:** Passing 100% of software tests verifies that the code behaves predictably and securely under valid and malformed inputs. **It does not prove that the underlying machine learning model is highly accurate, fair across all student demographic subgroups, or appropriate for consequential high-stakes decision-making.**
- **Generalization Performance Reality:** As established on Day 16, the champion Decision Tree achieved a test Macro F1 of **0.4498** and Accuracy of **0.5385** ($N=130$), reflecting noticeable class boundary uncertainty.
- **Probabilistic Early-Warning Tool:** Predictions must be treated as **exploratory early indicators**, never as certainties or deterministic judgments.
- **Non-Causal Associations:** The model captures statistical correlations in survey responses; it **does not demonstrate causal mechanisms**. Altering a feature does not guarantee an improvement in actual student learning.
- **Ethical Operational Guardrail:** The system must **never** be used as the sole basis for punitive actions, school tracking, or resource denial. Its sole intended purpose is to identify at-risk students who may benefit from proactive academic tutoring and counseling support.


---
### 17. Development Log

| Field | Entry / Details |
| :--- | :--- |
| **Day** | 18 |
| **Date** | 18 October 2026 |
| **Project** | AI-Based Student Performance Prediction System |
| **Internship** | YBI Foundation Python AI & Data Science Internship |
| **Primary Task** | Prediction workflow testing, validation, and user documentation |
| **Artifact Tested** | `artifacts/student_performance_pipeline.joblib` (Recovered from Day 17) |
| **Selected Model** | Decision Tree Classifier (`criterion='gini'`, `max_depth=10`, `min_samples_leaf=1`, `min_samples_split=10`) |
| **Test Scope** | Input validation, single/batch inference, unfamiliar categories, probability normalization, determinism, serialization |
| **Automated Tests** | 11 unit tests executed via `unittest` (11 passed, 0 failed, 0 errors, 100% pass rate) |
| **Outputs Saved** | `day_18_test_summary.csv`, `docs/Student_Performance_Prediction_User_Guide.md` |
| **Challenges Solved** | Verified dynamic class probability ordering alignment (`classes_`), tested unfamiliar category degradation without pipeline crash, implemented isolated temporary artifact testing. |
| **Next Steps** | Prepare a user-facing interactive demonstration or dashboard interface (Day 19), building upon the validated, documented pipeline. |


In [25]:
# ==============================================================================
# Final Comprehensive Validation Cell
# ==============================================================================
print("Starting Day 18 Final Comprehensive Validation Suite...")

# 1. Pipeline artifact exists and is fitted
assert pipeline_file.exists() and pipeline_file.stat().st_size > 0, "Pipeline artifact missing or empty!"
assert hasattr(fitted_pipeline, "classes_"), "Pipeline is not properly fitted!"
assert len(fitted_pipeline.feature_names_in_) == 30, "Pipeline feature count mismatch!"

# 2. Strict target leakage elimination
assert "G3" not in fitted_pipeline.feature_names_in_, "Target leakage: G3 present in pipeline features!"
assert "performance_category" not in fitted_pipeline.feature_names_in_, "Target leakage: target present in pipeline!"

# 3. Single and batch inference functionality
pred_single = predict_student_performance(fixture_valid_single, fitted_pipeline)
assert len(pred_single) == 1, "Single prediction count mismatch!"
pred_batch = predict_student_performance(fixture_valid_batch, fitted_pipeline)
assert len(pred_batch) == len(fixture_valid_batch), "Batch prediction count mismatch!"

# 4. Input validation catches malformed inputs
for _, bad_fixture, _ in validation_test_cases:
    try:
        validate_student_data(bad_fixture)
        raise AssertionError("Validation helper failed to raise ValueError on bad fixture!")
    except ValueError:
        pass  # Expected

# 5. Output labels and probabilities validity
for cat in pred_batch["predicted_category"]:
    assert cat in {"Low", "Medium", "High"}, f"Invalid output label: {cat}"
batch_probs = pred_batch[[c for c in pred_batch.columns if "proba" in c]].values
assert (batch_probs >= 0.0).all() and (batch_probs <= 1.0).all(), "Invalid probability range!"
assert np.allclose(batch_probs.sum(axis=1), 1.0, atol=1e-4), "Probability sum normalization failure!"

# 6. Automated test suite pass rate
assert total_passed == 11 and total_failures == 0 and total_errors == 0, "Automated tests did not pass completely!"
assert pass_rate_pct == 100.0, f"Expected 100% pass rate, got {pass_rate_pct}%"

# 7. Output files verification
assert test_summary_path.exists() and test_summary_path.stat().st_size > 0, "day_18_test_summary.csv missing or empty!"
assert user_guide_path.exists() and user_guide_path.stat().st_size > 500, "User guide missing or incomplete!"

print("=" * 70)
print("DAY 18 TESTING, VALIDATION, AND DOCUMENTATION COMPLETE!")
print("All 7 validation criteria passed with 100% compliance.")
print("Outputs verified on disk:")
print(f"  - Notebook:   notebooks/Day_18_Testing_Validation_Documentation.ipynb")
print(f"  - Test CSV:   {test_summary_path.name} ({test_summary_path.stat().st_size} bytes)")
print(f"  - User Guide: docs/{user_guide_path.name} ({user_guide_path.stat().st_size} bytes)")
print("=" * 70)


Starting Day 18 Final Comprehensive Validation Suite...
DAY 18 TESTING, VALIDATION, AND DOCUMENTATION COMPLETE!
All 7 validation criteria passed with 100% compliance.
Outputs verified on disk:
  - Notebook:   notebooks/Day_18_Testing_Validation_Documentation.ipynb
  - Test CSV:   day_18_test_summary.csv (2869 bytes)
  - User Guide: docs/Student_Performance_Prediction_User_Guide.md (872 bytes)
